# Assessment Outline
## Predicting Voter Turnout Changes in Virginia Census Tracts
### 2021 - 2025 Gubernatorial Elections

**Three Primary Methods**
* Feature Reduction of ACS data
* Regression (LASSO) to find the best columns
* Random Forest Classification

## Data Cleaning
### VA Counties + Cities 
* Import US Counties Shapefile
* Select those in Virginia
* Clean Names of Counties/Cities to Title Style

### Voter Turnout
* Group and Sum Turnout by County from Precincts
* Clean County Names to Title Style
* Join 2021 and 2025 on County Name
* Calculate Turnout Percent Change from 2021-2025

### ACS
* With 2024 data
    * Group Columns by County
    * Make County Names Title Style
* With 2021 data
    * Aquire only population estimate
    * Merge to ACS 2024 data
    * Calculate population percent change

### All
* Join all Datasets Together

In [1]:
import os

os.listdir()

['.ipynb_checkpoints',
 'ASGN_Prep.ipynb',
 'Data',
 'Template_submission_CASA0006.ipynb']

In [2]:
# Load Libraries
import pandas as pd
import geopandas as gpd


## Virginia Counties
counties = gpd.read_file("Data/Counties/cb_2022_us_county_500k.shp")
# select only virginia counties/cities and only the name and geometry
va_counties = counties[counties['STATE_NAME'] == 'Virginia'][['NAMELSAD', 'geometry']]
va_counties = va_counties.rename(columns={"NAMELSAD": "Locality"})

va_counties

,Locality,geometry
128,Middlesex County,"MULTIPOLYGON (((-76.42074 37.5976, -76.41951 3..."
129,Northampton County,"MULTIPOLYGON (((-75.98103 37.09714, -75.97802 ..."
130,Mecklenburg County,"POLYGON ((-78.74028 36.55127, -78.73654 36.552..."
131,Northumberland County,"POLYGON ((-76.63455 37.96854, -76.63243 37.971..."
132,Dickenson County,"POLYGON ((-82.55384 37.20284, -82.55037 37.204..."
...,...,...
3076,Fluvanna County,"POLYGON ((-78.49077 37.79759, -78.49057 37.797..."
3077,Manassas Park city,"POLYGON ((-77.4788 38.78258, -77.47359 38.7880..."
3104,Petersburg city,"POLYGON ((-77.45492 37.18762, -77.44646 37.194..."
3117,Wythe County,"POLYGON ((-81.37293 36.96332, -81.37053 36.965..."


In [3]:
## Voter Turnout
to_2021 = pd.read_csv('Data/2021_election_turnout.csv') 
to_2025 = pd.read_csv('Data/2025_election_turnout.csv')

# Remove local district types
to_2021 = to_2021[to_2021['district_type'] != 'Local']

# Group votes by county for 2021 and 2025
to_2021 = (
    to_2021
    .groupby("locality")["TotalVoteTurnout"]
    .sum()
    .reset_index()
)

to_2025 = (
    to_2025
    .groupby("locality")["TotalVoteTurnout"]
    .sum()
    .reset_index()
)

# Join the two new datasets
to_2021 = to_2021.rename(columns={"TotalVoteTurnout": "votes_2021"})
to_2025 = to_2025.rename(columns={"TotalVoteTurnout": "votes_2025"})
to_2021 = to_2021.rename(columns={"locality": "Locality"})
to_2025 = to_2025.rename(columns={"locality": "Locality"})
turnout = to_2021.merge(to_2025, on="Locality", how="inner")

# Create the percent change column
turnout["pct_change"] = (
    (turnout["votes_2025"] - turnout["votes_2021"]) 
    / turnout["votes_2021"]
) * 100
turnout

,Locality,votes_2021,votes_2025,pct_change
0,ACCOMACK COUNTY,12976,13150,1.340937
1,ALBEMARLE COUNTY,51575,56037,8.651478
2,ALEXANDRIA CITY,58608,63172,7.787333
3,ALLEGHANY COUNTY,6086,5826,-4.272100
4,AMELIA COUNTY,6372,6386,0.219711
...,...,...,...,...
128,WILLIAMSBURG CITY,4927,6755,37.101685
129,WINCHESTER CITY,8528,8857,3.857880
130,WISE COUNTY,11591,11127,-4.003106
131,WYTHE COUNTY,11616,11091,-4.519628


In [4]:
# Merge counties and votes
def normalize_locality(x):
    x = x.strip()
    x = x.replace("&", "And")
    x = x.title()
    x = x.replace(" County", " County")
    x = x.replace(" City", " City")
    return x

turnout["Locality"] = turnout["Locality"].str.title()

va_counties["Locality"] = va_counties["Locality"].apply(normalize_locality)
turnout["Locality"] = turnout["Locality"].apply(normalize_locality)

to_change = va_counties.merge(turnout, on="Locality", how="inner")
to_change

,Locality,geometry,votes_2021,votes_2025,pct_change
0,Middlesex County,"MULTIPOLYGON (((-76.42074 37.5976, -76.41951 3...",5631,5682,0.905701
1,Northampton County,"MULTIPOLYGON (((-75.98103 37.09714, -75.97802 ...",5287,5574,5.428409
2,Mecklenburg County,"POLYGON ((-78.74028 36.55127, -78.73654 36.552...",12080,11869,-1.746689
3,Northumberland County,"POLYGON ((-76.63455 37.96854, -76.63243 37.971...",6532,6878,5.296999
4,Dickenson County,"POLYGON ((-82.55384 37.20284, -82.55037 37.204...",4824,4482,-7.089552
...,...,...,...,...,...
128,Fluvanna County,"POLYGON ((-78.49077 37.79759, -78.49057 37.797...",12483,13504,8.179124
129,Manassas Park City,"POLYGON ((-77.4788 38.78258, -77.47359 38.7880...",3610,4016,11.246537
130,Petersburg City,"POLYGON ((-77.45492 37.18762, -77.44646 37.194...",9044,10067,11.311367
131,Wythe County,"POLYGON ((-81.37293 36.96332, -81.37053 36.965...",11616,11091,-4.519628


In [5]:
import numpy as np
## Load in ACS Data
demographics = pd.read_csv('Data/acs_dem.csv') 
econ = pd.read_csv('Data/acs_econ_charac.csv')
housing = pd.read_csv('Data/acs_housing.csv')
social = pd.read_csv('Data/acs_social_charac.csv')
acs_2021 = pd.read_csv('Data/acs_2021.csv')

## Drop the margin of error columns
# Define function to only keep the normalized data and the label column
def keeppct(df):
    cols = df.columns
    pct_mask = cols.str.contains(r'Percent$', case=False, regex=True)
    keep = pct_mask.copy()
    keep[0] = True
    return df.loc[:, keep]


# Apply the above function to the data
demographics = keeppct(demographics)
econ = keeppct(econ)
housing = keeppct(housing)
social = keeppct(social)
acs_2021 = keeppct(acs_2021)

## Clean the names of the columns
# define a function to change the column names
def cleancol(df):
    df = df.copy()
    df.columns = df.columns.str.replace(r'!!Percent$', '', regex=True)
    df.columns = df.columns.str.replace(r', Virginia$', '', regex=True)
    df.columns = df.columns.str.replace(r'\s*\(Grouping\)$', '', regex=True) 
    return df


# Apply second function
demographics = cleancol(demographics)
econ = cleancol(econ)
housing = cleancol(housing)
social = cleancol(social)
acs_2021 = cleancol(acs_2021)

## Remove all rows with NAN values
demographics = demographics.dropna()
econ = econ.dropna()
housing = housing.dropna()
social = social.dropna()
acs_2021 = acs_2021.dropna()

## Drop all rows with (X) values
def dropx(df):
    mask = ~(df == "(X)").any(axis=1)
    return df.loc[mask]

demographics = dropx(demographics)
econ = dropx(econ)
housing = dropx(housing)
social = dropx(social)
acs_2021 = dropx(acs_2021)

## Change orientation of data, i.e localities on the y axis
def switchsides(df):
    return df.set_index(df.columns[0]).T

demographics = switchsides(demographics)
econ = switchsides(econ)
housing = switchsides(housing)
social = switchsides(social)
acs_2021 = switchsides(acs_2021)

## Change the name of the Label Column to Locality
demographics.index.name = 'Locality'
econ.index.name = 'Locality'
housing.index.name = 'Locality'
social.index.name = 'Locality'
acs_2021.index.name = 'Locality'

## Aquire 2021 population
acs_2021 = acs_2021[[acs_2021.columns[0]]]
# remove duplicates
acs_2021 = acs_2021.loc[:, ~acs_2021.columns.duplicated()]
# change 2021 population column name
acs_2021.columns = acs_2021.columns.str.strip()
acs_2021 = acs_2021.rename(columns={'Total population': "2021_pop"})

## Join these 4 datasets based on the Locality attibute
attributes = pd.concat([acs_2021, demographics, econ, housing, social,], axis=1)

## Make a 2021-2024 population percent change column and delete the counts
# Strip white space in column names
attributes.columns = attributes.columns.str.strip()
# Remove duplicate columns
attributes = attributes.loc[:, ~attributes.columns.duplicated()]
# Convert strings to ints
attributes["Total population"] = pd.to_numeric(attributes["Total population"].str.replace(",", ""), errors="coerce")
attributes["2021_pop"] = pd.to_numeric(attributes["2021_pop"].str.replace(",", ""), errors="coerce")
# Calculate pct change
attributes['21-24_Pop_Pct_Change'] = (
    (attributes["Total population"] - attributes["2021_pop"]) / attributes["2021_pop"] * 100
)
# remove 2021_pop and total population columns
attributes = attributes.drop(columns=["Total population", "2021_pop"])
attributes

Label,Male,Female,Under 5 years,5 to 9 years,10 to 14 years,15 to 19 years,20 to 24 years,25 to 34 years,35 to 44 years,45 to 54 years,...,Slovak,Subsaharan African,Swedish,Swiss,Ukrainian,Welsh,West Indian (excluding Hispanic origin groups),With a computer,With a broadband Internet subscription,21-24_Pop_Pct_Change
Locality,,,,,,,,,,,,,,,,,,,,,
Accomack County,49.1%,50.9%,5.1%,4.7%,6.9%,6.2%,4.3%,10.7%,9.9%,11.2%,...,0.1%,0.3%,0.2%,0.1%,0.1%,0.3%,2.6%,89.2%,83.5%,-0.158740
Albemarle County,48.0%,52.0%,4.9%,5.4%,5.7%,8.9%,6.2%,12.5%,12.7%,11.0%,...,0.4%,0.6%,1.4%,0.5%,0.5%,0.9%,0.5%,97.1%,92.9%,3.123710
Alleghany County,49.6%,50.4%,3.6%,4.0%,6.8%,5.8%,5.4%,10.8%,9.6%,12.1%,...,0.1%,0.1%,0.1%,0.2%,0.1%,0.1%,0.0%,87.9%,82.8%,-2.666055
Amelia County,49.3%,50.7%,5.2%,3.8%,6.6%,6.0%,5.2%,11.6%,12.2%,12.1%,...,0.1%,1.1%,0.2%,0.1%,0.0%,5.4%,0.0%,91.7%,84.2%,1.830560
Amherst County,48.4%,51.6%,4.5%,6.4%,5.7%,6.4%,5.8%,11.1%,11.6%,11.6%,...,0.0%,0.8%,0.4%,0.2%,0.1%,0.5%,0.1%,90.8%,84.5%,0.331411
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Suffolk city,48.6%,51.4%,6.3%,6.1%,7.1%,6.2%,5.7%,13.4%,14.0%,12.3%,...,0.1%,5.7%,0.7%,0.2%,0.1%,0.5%,1.0%,95.4%,90.2%,5.927006
Virginia Beach city,49.0%,51.0%,5.8%,6.1%,6.2%,5.9%,6.3%,15.4%,14.3%,11.6%,...,0.2%,1.6%,0.6%,0.2%,0.3%,0.6%,1.1%,97.3%,94.4%,-0.286021
Waynesboro city,48.4%,51.6%,6.5%,7.8%,4.2%,5.2%,5.6%,15.0%,14.2%,11.0%,...,0.2%,0.0%,0.6%,0.2%,0.0%,0.9%,0.2%,94.9%,91.4%,3.124295


In [6]:
## Join the vote data and attributes together
# Set index of to_change to Locality
to_change.set_index("Locality", inplace=True)

In [7]:
to_change

,geometry,votes_2021,votes_2025,pct_change
Locality,,,,
Middlesex County,"MULTIPOLYGON (((-76.42074 37.5976, -76.41951 3...",5631,5682,0.905701
Northampton County,"MULTIPOLYGON (((-75.98103 37.09714, -75.97802 ...",5287,5574,5.428409
Mecklenburg County,"POLYGON ((-78.74028 36.55127, -78.73654 36.552...",12080,11869,-1.746689
Northumberland County,"POLYGON ((-76.63455 37.96854, -76.63243 37.971...",6532,6878,5.296999
Dickenson County,"POLYGON ((-82.55384 37.20284, -82.55037 37.204...",4824,4482,-7.089552
...,...,...,...,...
Fluvanna County,"POLYGON ((-78.49077 37.79759, -78.49057 37.797...",12483,13504,8.179124
Manassas Park City,"POLYGON ((-77.4788 38.78258, -77.47359 38.7880...",3610,4016,11.246537
Petersburg City,"POLYGON ((-77.45492 37.18762, -77.44646 37.194...",9044,10067,11.311367


In [8]:
attributes.index = attributes.index.to_series().apply(normalize_locality)

In [9]:
# Merge to_change with attributes
data = to_change.join(attributes, how="inner")
data

,geometry,votes_2021,votes_2025,pct_change,Male,Female,Under 5 years,5 to 9 years,10 to 14 years,15 to 19 years,...,Slovak,Subsaharan African,Swedish,Swiss,Ukrainian,Welsh,West Indian (excluding Hispanic origin groups),With a computer,With a broadband Internet subscription,21-24_Pop_Pct_Change
Locality,,,,,,,,,,,,,,,,,,,,,
Middlesex County,"MULTIPOLYGON (((-76.42074 37.5976, -76.41951 3...",5631,5682,0.905701,50.0%,50.0%,3.5%,5.1%,4.8%,3.5%,...,0.5%,0.1%,0.3%,0.1%,0.1%,1.0%,0.0%,91.6%,84.3%,1.506644
Northampton County,"MULTIPOLYGON (((-75.98103 37.09714, -75.97802 ...",5287,5574,5.428409,48.7%,51.3%,4.0%,5.8%,5.7%,4.3%,...,0.1%,0.6%,0.3%,0.2%,0.0%,0.2%,0.3%,91.3%,85.5%,-1.365941
Mecklenburg County,"POLYGON ((-78.74028 36.55127, -78.73654 36.552...",12080,11869,-1.746689,48.9%,51.1%,4.9%,4.6%,5.5%,5.2%,...,0.1%,3.0%,0.1%,0.0%,0.1%,0.6%,0.2%,80.6%,74.7%,0.556892
Northumberland County,"POLYGON ((-76.63455 37.96854, -76.63243 37.971...",6532,6878,5.296999,47.7%,52.3%,4.0%,2.0%,4.9%,4.8%,...,0.3%,1.9%,0.2%,0.3%,0.3%,0.8%,0.2%,89.3%,81.1%,1.863769
Dickenson County,"POLYGON ((-82.55384 37.20284, -82.55037 37.204...",4824,4482,-7.089552,51.2%,48.8%,4.4%,3.8%,7.3%,4.8%,...,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%,0.0%,89.2%,83.5%,-3.668631
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Fluvanna County,"POLYGON ((-78.49077 37.79759, -78.49057 37.797...",12483,13504,8.179124,46.1%,53.9%,5.2%,5.9%,4.9%,5.2%,...,0.0%,0.7%,1.7%,0.2%,1.0%,0.9%,0.1%,97.4%,93.8%,3.744737
Manassas Park City,"POLYGON ((-77.4788 38.78258, -77.47359 38.7880...",3610,4016,11.246537,50.7%,49.3%,5.9%,4.9%,8.1%,5.8%,...,0.0%,2.6%,0.2%,0.0%,0.1%,0.3%,0.4%,98.8%,92.6%,-1.656812
Petersburg City,"POLYGON ((-77.45492 37.18762, -77.44646 37.194...",9044,10067,11.311367,45.2%,54.8%,8.0%,8.1%,5.0%,4.4%,...,0.0%,3.5%,0.3%,0.1%,0.1%,0.0%,1.5%,92.9%,85.5%,0.926901


In [13]:
## Export file as a geojson
# Conver to gpd
data = gpd.GeoDataFrame(
    data,
    geometry="geometry",
    crs="EPSG:2283"   # Meters based Virginia Specific CRS (NAD 83/Virginia North)
)
data.to_file("Data/total_data.geojson", driver="GeoJSON")